Q1. Embedding a query

In [9]:
from embedder import Embedder

embedder = Embedder()

query = "How does approximate nearest neighbor search work?"

v = embedder.encode(query)
len(v)

384

In [10]:
v[0]

np.float64(-0.02058203437252893)

Loading the data from the lesson pages from the course repository.

In [11]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
len(documents)

72

In [12]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

Q2. Cosine similarity

In [13]:
page_name = "02-vector-search/lessons/07-sqlitesearch-vector.md"

page = [doc for doc in documents if doc['filename'] == page_name]
page

[{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done

In [14]:
v_page = embedder.encode(page[0]["content"])
similarity = v.dot(v_page)
similarity

np.float64(0.36107027225589694)

Q3. Chunking and search by hand

In [15]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [16]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [17]:
len(chunks)

295

In [18]:
X = embedder.encode_batch([c["content"] for c in chunks])

In [19]:
X.shape

(295, 384)

In [20]:
scores = X.dot(v)

In [21]:
scores.shape

(295,)

In [22]:
best = chunks[scores.argmax()]

In [23]:
best["filename"]


'02-vector-search/lessons/07-sqlitesearch-vector.md'

Q4. Vector search with minsearch

In [24]:
X.shape

(295, 384)

In [25]:
len(documents)

72

In [26]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, chunks)

In [27]:
q2 = "What metric do we use to evaluate a search engine?"
v2 = embedder.encode(q2)

results = vindex.search(v2, num_results=5)

In [28]:
results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

Q5. Text search vs vector search

In [29]:
q3 = "How do I store vectors in PostgreSQL?"
v3 = embedder.encode(q3)

In [30]:
from minsearch import Index
index = Index(text_fields=["content"],keyword_fields=["filename"])
index.fit(chunks)

In [31]:
text_search_result = index.search(q3, num_results=5)

In [32]:
v_search_results = vindex.search(v3, num_results=5)

In [33]:
for r in text_search_result:
    print(r["filename"])

02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


In [34]:
for r in v_search_results:
    print(r["filename"])

02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


Q6. Hybrid search

Every document scores by its position (rank, starting at 0) in each list, and we sum the scores across lists with a constant k = 60:

RRF(d) = sum over lists of  1 / (k + rank(d))

In [35]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [36]:
q4 = "How do I give the model access to tools?"
v4 = embedder.encode(q4)

v_results = vindex.search(v4, num_results =5)
t_results = index.search(q4, num_results=5)

In [38]:
results = rrf([v_results, t_results])

In [41]:
results[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'

In [42]:
v_results[0]

{'start': 2000,
 'content': 'wrong.\n\n## The project\n\nRAG solves these problems by giving the LLM relevant documents at\nquestion time. We don\'t hope the model memorized the answer. We\nretrieve the right information and hand it to the LLM, and the model\ngenerates a grounded response. This lets us inject knowledge the model\nnever saw during training. That\'s why RAG is still the most common way\npeople use LLMs in the industry.\n\nTo make this concrete, we build a FAQ agent for our course. A student\nasks something like "when does the course start?" and the agent answers\nfrom the FAQ data we prepared.\n\nThis module has two parts.\n\nIn Part 1 (the next 9 lessons) we will:\n\n- Understand what RAG is and how it works\n- Build a search engine over a real FAQ dataset\n- Write a prompt that combines the user\'s question with search results\n- Wire it all together into a working RAG pipeline\n- Split ingestion and query into separate processes\n\nIn Part 2, we make the pipeline agen

In [43]:
t_results[0]

{'start': 0,
 'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can c